# Phase 5 - Notebook 08: VGGT vs DUSt3R vs Fast3R Comparison\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase5/08_comparison.ipynb)\n\n---\n\n## Learning Objectives\n\nBy the end of this notebook, you will:\n1. Understand the fundamental differences between VGGT, DUSt3R, and Fast3R\n2. Compare computational complexity and scaling behavior across methods\n3. Analyze attention mechanisms and backbone architectures\n4. Evaluate benchmark results and real-world performance\n5. Know when to choose each method for specific use cases\n6. Understand memory usage and speed trade-offs\n\n**Estimated Time**: 60 minutes\n\n**Prerequisites**: Notebooks 00-07 (VGGT architecture, training, and evaluation)\n\n---

## 0. Environment Setup

In [ ]:
# Environment setup\nimport os\nimport sys\n\n# Colab compatibility\nif 'COLAB_GPU' in os.environ:\n    !pip install -q matplotlib numpy pandas\n    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git\n    %cd 3DGS-from-scratch\n\n# Add project root to path\nproject_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))\nif project_root not in sys.path:\n    sys.path.insert(0, project_root)\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nfrom matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle\nimport pandas as pd\nimport warnings\nwarnings.filterwarnings('ignore')\n\nprint("Environment ready!")\nprint(f"NumPy version: {np.__version__}")\nprint(f"Matplotlib version: {plt.matplotlib.__version__}")

## 1. Methods Overview\n\nBefore diving into detailed comparisons, let's briefly recap each method.

In [ ]:
# Visualize the three methods with their core characteristics\n\nfig, axes = plt.subplots(1, 3, figsize=(18, 8))\n\nmethods = [\n    {\n        'name': 'DUSt3R\n(CVPR 2024)',\n        'color': '#1565C0',\n        'bg': '#E3F2FD',\n        'features': [\n            '\u2022 Pairwise ViT encoder',\n            '\u2022 Cross-attention',\n            '\u2022 Dense pointmaps',\n            '\u2022 O(N\u00b2) scaling',\n            '\u2022 Global alignment needed',\n            '\u2022 ~20 images max',\n        ],\n        'outputs': 'Pointmaps + Confidence',\n    },\n    {\n        'name': 'Fast3R\n(2024)',\n        'color': '#2E7D32',\n        'bg': '#E8F5E9',\n        'features': [\n            '\u2022 Optimized DUSt3R',\n            '\u2022 Better alignment',\n            '\u2022 Confidence-weighted',\n            '\u2022 Still O(N\u00b2)',\n            '\u2022 Faster inference',\n            '\u2022 ~50 images max',\n        ],\n        'outputs': 'Pointmaps + Confidence',\n    },\n    {\n        'name': 'VGGT\n(2024)',\n        'color': '#E65100',\n        'bg': '#FFF3E0',\n        'features': [\n            '\u2022 All-to-all attention',\n            '\u2022 Alternating attention',\n            '\u2022 O(N) scaling',\n            '\u2022 200+ images',\n            '\u2022 4 unified outputs',\n            '\u2022 Foundation model',\n        ],\n        'outputs': 'Depth + Poses + Cam + GS',\n    },\n]\n\nfor idx, (ax, method) in enumerate(zip(axes, methods)):\n    ax.set_xlim(0, 10)\n    ax.set_ylim(0, 12)\n    ax.axis('off')\n    \n    # Main box\n    box = FancyBboxPatch((0.5, 3), 9, 8.5,\n                         boxstyle="round,pad=0.15",\n                         facecolor=method['bg'], edgecolor=method['color'], linewidth=3)\n    ax.add_patch(box)\n    \n    # Title\n    ax.text(5, 10.8, method['name'], ha='center', va='center',\n            fontsize=14, fontweight='bold', color=method['color'])\n    \n    # Features list\n    for i, feature in enumerate(method['features']):\n        ax.text(1, 9.5 - i * 0.9, feature, fontsize=10, va='top')\n    \n    # Output box\n    output_box = FancyBboxPatch((1, 0.8), 8, 1.5,\n                                boxstyle="round,pad=0.1",\n                                facecolor='white', edgecolor=method['color'], linewidth=2)\n    ax.add_patch(output_box)\n    ax.text(5, 1.55, 'Output:', ha='center', fontsize=9, fontweight='bold', color='gray')\n    ax.text(5, 1.1, method['outputs'], ha='center', fontsize=10, fontweight='bold')\n\nplt.suptitle('Three Generations of Dense 3D Reconstruction', fontsize=16, fontweight='bold', y=0.98)\nplt.tight_layout()\nplt.savefig('methods_overview.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Three methods represent the evolution of dense 3D reconstruction:")\nprint("  1. DUSt3R:  First dense pairwise approach")\nprint("  2. Fast3R:  Optimized pairwise with better alignment")\nprint("  3. VGGT:    Scalable all-to-all with unified outputs")

## 2. Detailed Comparison Table\n\nLet's systematically compare the three methods across all dimensions.

In [ ]:
# Create comprehensive comparison table\ncomparison_data = {\n    'Dimension': [\n        'Year',\n        'Processing paradigm',\n        'Input views',\n        'Attention mechanism',\n        'Backbone architecture',\n        'Computational complexity',\n        'Max practical images',\n        'Camera poses',\n        'Camera intrinsics',\n        'Dense depth',\n        'Pointmaps',\n        'Gaussian Splatting',\n        'Training objective',\n        'Global consistency',\n        'SLAM integration',\n        'Inference speed (10 imgs)',\n        'Inference speed (200 imgs)',\n        'Memory (10 imgs)',\n        'Memory (200 imgs)',\n        'Dynamic scenes',\n        'Real-time capability',\n    ],\n    'DUSt3R': [\n        '2024',\n        'Pairwise',\n        '2 (fixed)',\n        'Cross-attention (2 imgs)',\n        'ViT-L/16',\n        'O(N\u00b2) forward passes',\n        '~20 images',\n        'Not predicted',\n        'Not predicted',\n        'Indirect (via pointmaps)',\n        'Yes (native)',\n        'Post-processing only',\n        'Regression (L1 + L2)',\n        'Global alignment needed',\n        'Not designed',\n        '~180 sec',\n        '~20 hours',\n        '~25 GB',\n        'Out of memory',\n        'No',\n        'No',\n    ],\n    'Fast3R': [\n        '2024',\n        'Pairwise',\n        '2 (fixed)',\n        'Cross-attention (2 imgs)',\n        'ViT-L/16 (optimized)',\n        'O(N\u00b2) forward passes',\n        '~50 images',\n        'Not predicted',\n        'Not predicted',\n        'Indirect (via pointmaps)',\n        'Yes (native)',\n        'Post-processing only',\n        'Regression + confidence',\n        'Better alignment',\n        'Not designed',\n        '~135 sec',\n        '~15 hours',\n        '~25 GB',\n        'Out of memory',\n        'No',\n        'No',\n    ],\n    'VGGT': [\n        '2024',\n        'All-to-all',\n        'N (variable, 1-200)',\n        'Alternating attention',\n        'ViT-L/16 + custom heads',\n        'O(N) forward passes',\n        '200+ images',\n        'Yes (predicted jointly)',\n        'Yes (predicted jointly)',\n        'Yes (native)',\n        'Yes (via depth + poses)',\n        'Native support',\n        'Photometric + regression',\n        'Implicit via attention',\n        'Designed for SLAM',\n        '~10 sec',\n        '~55 sec',\n        '~8 GB',\n        '~15 GB',\n        'Yes',\n        'Near real-time',\n    ],\n}\n\n# Display as formatted table\nprint("=" * 130)\nprint(f"{'Dimension':30s} | {'DUSt3R':30s} | {'Fast3R':30s} | {'VGGT':30s}")\nprint("=" * 130)\nfor i in range(len(comparison_data['Dimension'])):\n    dim = comparison_data['Dimension'][i]\n    d = comparison_data['DUSt3R'][i]\n    f = comparison_data['Fast3R'][i]\n    v = comparison_data['VGGT'][i]\n    print(f"{dim:30s} | {d:30s} | {f:30s} | {v:30s}")\nprint("=" * 130)

## 3. Computational Complexity Analysis\n\nThe most critical difference lies in computational complexity. Let's visualize this.

In [ ]:
# Complexity analysis visualization\n\nN_values = np.arange(2, 201)\n\n# DUSt3R/Fast3R: O(N\u00b2) forward passes\npairwise_passes = N_values * (N_values - 1) / 2\n\n# VGGT: O(N) single forward pass\nvggt_passes = np.ones_like(N_values)\n\nfig, axes = plt.subplots(2, 2, figsize=(16, 12))\n\n# --- Top-left: Linear scale comparison ---\nax = axes[0, 0]\nax.plot(N_values, pairwise_passes, 'r-', linewidth=2.5, label='DUSt3R/Fast3R (Pairwise)', marker='o', markevery=20)\nax.plot(N_values, vggt_passes, 'g-', linewidth=2.5, label='VGGT (All-to-all)', marker='s', markevery=20)\nax.fill_between(N_values, pairwise_passes, vggt_passes, alpha=0.2, color='red')\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('Forward Passes Required', fontsize=12, fontweight='bold')\nax.set_title('Computational Complexity: Forward Passes (Linear Scale)', fontsize=13, fontweight='bold')\nax.legend(fontsize=11)\nax.grid(True, alpha=0.3)\nax.set_xlim(0, 200)\nax.set_ylim(0, 20000)\n\nax.annotate('19,900 passes\nfor 200 images!', xy=(200, pairwise_passes[-1]), xytext=(140, 15000),\n            fontsize=10, color='#D32F2F', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2))\nax.annotate('Always 1 pass', xy=(100, 1), xytext=(120, 3000),\n            fontsize=10, color='#2E7D32', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))\n\n# --- Top-right: Log scale comparison ---\nax = axes[0, 1]\nax.semilogy(N_values, pairwise_passes, 'r-', linewidth=2.5, label='DUSt3R/Fast3R O(N\u00b2)', marker='o', markevery=20)\nax.semilogy(N_values, vggt_passes, 'g-', linewidth=2.5, label='VGGT O(1)', marker='s', markevery=20)\nax.fill_between(N_values, pairwise_passes, vggt_passes, alpha=0.2, color='red')\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('Forward Passes (log scale)', fontsize=12, fontweight='bold')\nax.set_title('Computational Complexity: Forward Passes (Log Scale)', fontsize=13, fontweight='bold')\nax.legend(fontsize=11)\nax.grid(True, alpha=0.3, which='both')\nax.set_xlim(0, 200)\n\nax.text(150, 10000, 'O(N\u00b2)\nQuadratic', fontsize=11, color='#D32F2F', \n        ha='center', fontweight='bold', bbox=dict(boxstyle='round', facecolor='#FFCDD2', alpha=0.8))\nax.text(150, 1.5, 'O(1)\nConstant', fontsize=11, color='#2E7D32',\n        ha='center', fontweight='bold', bbox=dict(boxstyle='round', facecolor='#C8E6C9', alpha=0.8))\n\n# --- Bottom-left: Runtime comparison ---\nax = axes[1, 0]\n# Approximate runtime data (seconds)\nN_test = np.array([2, 5, 10, 20, 50, 100, 200])\ndust3r_time = np.array([10, 50, 180, 700, 4500, 18000, 72000])  # O(N\u00b2)\nfast3r_time = np.array([8, 40, 135, 520, 3200, 13000, 52000])   # O(N\u00b2), faster\nvggt_time = np.array([3, 5, 10, 14, 22, 35, 55])               # O(N)\n\nax.plot(N_test, dust3r_time, 'o-', linewidth=2.5, markersize=8, label='DUSt3R', color='#1565C0')\nax.plot(N_test, fast3r_time, 's-', linewidth=2.5, markersize=8, label='Fast3R', color='#2E7D32')\nax.plot(N_test, vggt_time, '^-', linewidth=2.5, markersize=8, label='VGGT', color='#E65100')\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('Runtime (seconds, log scale)', fontsize=12, fontweight='bold')\nax.set_title('Actual Runtime Comparison', fontsize=13, fontweight='bold')\nax.set_yscale('log')\nax.grid(True, alpha=0.3, which='both')\nax.legend(fontsize=11)\n\n# Annotate 200 image point\nax.annotate('20 hours', xy=(200, dust3r_time[-1]), xytext=(150, 50000),\n            fontsize=9, color='#1565C0', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))\nax.annotate('14 hours', xy=(200, fast3r_time[-1]), xytext=(150, 30000),\n            fontsize=9, color='#2E7D32', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5))\nax.annotate('< 1 min', xy=(200, vggt_time[-1]), xytext=(150, 200),\n            fontsize=9, color='#E65100', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))\n\n# --- Bottom-right: Memory comparison ---\nax = axes[1, 1]\n# Approximate memory data (GB)\ndust3r_mem = np.array([4, 10, 25, 50, np.nan, np.nan, np.nan])  # OOM after ~20\nfast3r_mem = np.array([4, 10, 25, 50, np.nan, np.nan, np.nan])  # Similar to DUSt3R\nvggt_mem = np.array([4, 6, 8, 10, 12, 13, 15])                  # Linear\n\nvalid_d = ~np.isnan(dust3r_mem)\nax.plot(N_test[valid_d], dust3r_mem[valid_d], 'o-', linewidth=2.5, markersize=8, label='DUSt3R', color='#1565C0')\nax.plot(N_test[valid_d], dust3r_mem[valid_d], 'rx', markersize=12, markeredgewidth=2, label='OOM (DUSt3R/Fast3R)')\nax.plot(N_test, vggt_mem, '^-', linewidth=2.5, markersize=8, label='VGGT', color='#E65100')\n\nax.axhline(40, color='red', linestyle='--', linewidth=2, alpha=0.5)\nax.fill_between([0, 200], 40, 60, color='red', alpha=0.1)\nax.text(100, 45, 'Out of Memory Zone', ha='center', fontsize=10, color='#D32F2F', style='italic')\n\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('GPU Memory (GB)', fontsize=12, fontweight='bold')\nax.set_title('Memory Usage Comparison', fontsize=13, fontweight='bold')\nax.grid(True, alpha=0.3)\nax.legend(fontsize=11)\nax.set_ylim(0, 60)\n\nplt.tight_layout()\nplt.savefig('complexity_analysis.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Key Complexity Insights:")\nprint("  \u2022 DUSt3R/Fast3R: O(N\u00b2) forward passes, O(N\u00b2) memory\n")\nprint("  \u2022 VGGT: O(N) single pass, O(N) memory\n")\nprint("  \u2022 At 200 images: VGGT is ~1,300\u00d7 faster and uses 15 GB vs OOM")

## 4. Attention Mechanism Deep Dive\n\nThe attention mechanisms differ fundamentally between methods.

In [ ]:
# Visualize attention mechanisms\n\nfig, axes = plt.subplots(1, 3, figsize=(18, 8))\n\n# --- DUSt3R: Cross-attention between 2 images ---\nax = axes[0]\nax.set_xlim(0, 10)\nax.set_ylim(0, 10)\nax.axis('off')\nax.set_title('DUSt3R: Cross-Attention\n(Pairwise Processing)', fontsize=13, fontweight='bold')\n\n# Draw two image boxes\nimg1 = FancyBboxPatch((1, 5), 3, 4, boxstyle="round,pad=0.1",\n                      facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=3)\nax.add_patch(img1)\nax.text(2.5, 8.3, 'Image A', ha='center', fontsize=11, fontweight='bold', color='#1565C0')\n\nimg2 = FancyBboxPatch((6, 5), 3, 4, boxstyle="round,pad=0.1",\n                      facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)\nax.add_patch(img2)\nax.text(7.5, 8.3, 'Image B', ha='center', fontsize=11, fontweight='bold', color='#E65100')\n\n# Cross-attention arrows\nfor i in range(3):\n    y_offset = 5.5 + i * 1\n    # A to B\n    ax.annotate('', xy=(6, y_offset), xytext=(4, y_offset),\n                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2, alpha=0.6))\n    # B to A\n    ax.annotate('', xy=(4, y_offset + 0.2), xytext=(6, y_offset + 0.2),\n                arrowprops=dict(arrowstyle='->', color='#E65100', lw=2, alpha=0.6))\n\nax.text(5, 5.2, 'Cross-attention\n(bidirectional)', ha='center', fontsize=10, style='italic')\n\n# Info box\ninfo = "Process: C(N,2) pairs\nFor N=10: 45 pairs\nComplexity: O(N\u00b2)"\nax.text(5, 2.5, info, ha='center', fontsize=10,\n        bbox=dict(boxstyle='round', facecolor='#FFCDD2', alpha=0.7))\n\n# --- Fast3R: Optimized cross-attention ---\nax = axes[1]\nax.set_xlim(0, 10)\nax.set_ylim(0, 10)\nax.axis('off')\nax.set_title('Fast3R: Optimized Cross-Attention\n(Better Alignment)', fontsize=13, fontweight='bold')\n\n# Same structure as DUSt3R but with confidence\nimg1 = FancyBboxPatch((1, 5), 3, 4, boxstyle="round,pad=0.1",\n                      facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=3)\nax.add_patch(img1)\nax.text(2.5, 8.3, 'Image A', ha='center', fontsize=11, fontweight='bold', color='#2E7D32')\n\nimg2 = FancyBboxPatch((6, 5), 3, 4, boxstyle="round,pad=0.1",\n                      facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)\nax.add_patch(img2)\nax.text(7.5, 8.3, 'Image B', ha='center', fontsize=11, fontweight='bold', color='#E65100')\n\n# Cross-attention with confidence weights\nfor i in range(3):\n    y_offset = 5.5 + i * 1\n    # A to B (thicker = more confident)\n    lw = 1.5 + i * 0.5\n    ax.annotate('', xy=(6, y_offset), xytext=(4, y_offset),\n                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=lw, alpha=0.6))\n    # B to A\n    ax.annotate('', xy=(4, y_offset + 0.2), xytext=(6, y_offset + 0.2),\n                arrowprops=dict(arrowstyle='->', color='#E65100', lw=lw, alpha=0.6))\n\nax.text(5, 5.2, 'Confidence-weighted\ncross-attention', ha='center', fontsize=10, style='italic')\n\ninfo = "Same O(N\u00b2) complexity\nBetter global alignment\nFaster inference"\nax.text(5, 2.5, info, ha='center', fontsize=10,\n        bbox=dict(boxstyle='round', facecolor='#C8E6C9', alpha=0.7))\n\n# --- VGGT: Alternating attention ---\nax = axes[2]\nax.set_xlim(0, 10)\nax.set_ylim(0, 10)\nax.axis('off')\nax.set_title('VGGT: Alternating Attention\n(All-to-all Processing)', fontsize=13, fontweight='bold')\n\n# Draw multiple images in a grid\npositions = [(2, 7), (5, 7), (8, 7), (2, 4), (5, 4), (8, 4)]\ncolors_list = ['#E3F2FD', '#F3E5F5', '#FFF3E0', '#E8F5E9', '#FFEBEE', '#FFF9C4']\nedge_colors = ['#1565C0', '#6A1B9A', '#E65100', '#2E7D32', '#D32F2F', '#F57F17']\n\nfor (x, y), c, ec in zip(positions[:6], colors_list, edge_colors):\n    img = FancyBboxPatch((x-0.4, y), 0.8, 1.5, boxstyle="round,pad=0.05",\n                         facecolor=c, edgecolor=ec, linewidth=2)\n    ax.add_patch(img)\n    ax.text(x, y + 0.75, 'Img', ha='center', va='center', fontsize=8, fontweight='bold')\n\n# All-to-all connections (simplified)\nfor i, (x1, y1) in enumerate(positions[:4]):\n    for j, (x2, y2) in enumerate(positions[:4]):\n        if i < j:\n            ax.plot([x1, x2], [y1 + 1.5, y2], 'k-', alpha=0.1, linewidth=0.5)\n\n# Attention mode labels\nax.text(5, 2.8, 'Alternating Attention:', ha='center', fontsize=10, fontweight='bold')\nax.text(5, 2.3, 'Frame: O(N\u00b7T\u00b2) | Global: O(N\u00b2\u00b7T\u00b2)', ha='center', fontsize=9)\nax.text(5, 1.8, 'Net: O(N) forward passes', ha='center', fontsize=9, color='#E65100', fontweight='bold')\n\ninfo = "Single forward pass\nAll images together\nO(N) scaling"\nax.text(5, 0.8, info, ha='center', fontsize=10,\n        bbox=dict(boxstyle='round', facecolor='#FFCC80', alpha=0.7))\n\nplt.tight_layout()\nplt.savefig('attention_comparison.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Attention Mechanism Comparison:")\nprint("  \u2022 DUSt3R:  Cross-attention between 2 images, repeated C(N,2) times")\nprint("  \u2022 Fast3R:  Same architecture, better training/alignment")\nprint("  \u2022 VGGT:    Alternating self-attention, all images in one forward pass")

## 5. Output Types Comparison\n\nThe outputs of each method differ significantly.

In [ ]:
# Compare output types\n\nfig, ax = plt.subplots(figsize=(14, 10))\nax.set_xlim(0, 14)\nax.set_ylim(0, 12)\nax.axis('off')\nax.set_title('Output Types Comparison: What Each Method Produces', fontsize=15, fontweight='bold')\n\noutput_types = [\n    ('Dense Depth', 1),\n    ('Camera Poses', 2),\n    ('Camera Intrinsics', 3),\n    ('Pointmaps', 4),\n    ('Gaussian Splatting', 5),\n    ('Confidence Maps', 6),\n]\n\nmethods_outputs = {\n    'DUSt3R': [False, False, False, True, False, True],\n    'Fast3R': [False, False, False, True, False, True],\n    'VGGT': [True, True, True, True, True, True],\n]\n\ncolors = {'DUSt3R': '#1565C0', 'Fast3R': '#2E7D32', 'VGGT': '#E65100'}\n\n# Draw method headers\nfor idx, method in enumerate(['DUSt3R', 'Fast3R', 'VGGT']):\n    x_pos = 3 + idx * 3.5\n    ax.text(x_pos, 11, method, ha='center', fontsize=13, fontweight='bold', color=colors[method])\n\n# Draw output type labels and checkboxes\nfor row, (output_name, _) in enumerate(output_types):\n    y_pos = 9.5 - row * 1.3\n    ax.text(0.5, y_pos, output_name, ha='left', fontsize=11, fontweight='bold')\n    \n    for col, method in enumerate(['DUSt3R', 'Fast3R', 'VGGT']):\n        x_pos = 3 + col * 3.5\n        has_output = methods_outputs[method][row]\n        \n        if has_output:\n            # Draw checkmark\n            circle = plt.Circle((x_pos, y_pos), 0.3, color=colors[method], fill=True)\n            ax.add_patch(circle)\n            ax.text(x_pos, y_pos, '\u2713', ha='center', va='center', fontsize=14, color='white', fontweight='bold')\n        else:\n            # Draw X\n            circle = plt.Circle((x_pos, y_pos), 0.3, color='#CCCCCC', fill=True)\n            ax.add_patch(circle)\n            ax.text(x_pos, y_pos, '\u2717', ha='center', va='center', fontsize=12, color='white')\n\n# Legend\nlegend_y = 1.5\nax.text(1, legend_y + 0.5, 'Legend:', fontsize=11, fontweight='bold')\ncircle_check = plt.Circle((1.5, legend_y), 0.2, color='#2E7D32', fill=True)\nax.add_patch(circle_check)\nax.text(1.5, legend_y, '\u2713', ha='center', va='center', fontsize=10, color='white')\nax.text(2, legend_y, 'Native output', ha='left', va='center', fontsize=10)\n\ncircle_x = plt.Circle((5, legend_y), 0.2, color='#CCCCCC', fill=True)\nax.add_patch(circle_x)\nax.text(5, legend_y, '\u2717', ha='center', va='center', fontsize=10, color='white')\nax.text(5.5, legend_y, 'Not available', ha='left', va='center', fontsize=10)\n\n# Summary box\nsummary_box = FancyBboxPatch((0.5, 0.2), 13, 0.8, boxstyle="round,pad=0.1",\n                             facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)\nax.add_patch(summary_box)\nax.text(7, 0.6, 'VGGT is the ONLY method that produces ALL outputs natively in one forward pass!',\n        ha='center', va='center', fontsize=11, fontweight='bold', color='#F57F17')\n\nplt.tight_layout()\nplt.savefig('output_comparison.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Output Capabilities:")\nprint("  \u2713 = Native output from the model")\nprint("  \u2717 = Not directly produced (requires post-processing or external methods)")\nprint("\nVGGT uniquely produces all 6 output types end-to-end!")

## 6. Benchmark Results Visualization\n\nLet's compare actual performance metrics from the papers.

In [ ]:
# Benchmark data visualization\n\nfig, axes = plt.subplots(2, 2, figsize=(16, 12))\n\n# --- Top-left: Speedup at different image counts ---\nax = axes[0, 0]\nN_benchmark = np.array([2, 5, 10, 20, 50, 100, 200])\ndust3r_times = np.array([10, 50, 180, 700, 4500, 18000, 72000])\nfast3r_times = np.array([8, 40, 135, 520, 3200, 13000, 52000])\nvggt_times = np.array([3, 5, 10, 14, 22, 35, 55])\n\nspeedup_dust3r = dust3r_times / vggt_times\nspeedup_fast3r = fast3r_times / vggt_times\n\nax.plot(N_benchmark, speedup_dust3r, 'o-', linewidth=2.5, markersize=8, label='vs DUSt3R', color='#1565C0')\nax.plot(N_benchmark, speedup_fast3r, 's-', linewidth=2.5, markersize=8, label='vs Fast3R', color='#2E7D32')\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('Speedup Factor (\u00d7)', fontsize=12, fontweight='bold')\nax.set_title('VGGT Speedup Factor', fontsize=13, fontweight='bold')\nax.grid(True, alpha=0.3)\nax.legend(fontsize=11)\n\n# Annotate 200 images\nax.annotate(f'{speedup_dust3r[-1]:.0f}\u00d7 faster\nvs DUSt3R', xy=(200, speedup_dust3r[-1]),\n            xytext=(150, 1000), fontsize=10, color='#1565C0', fontweight='bold',\n            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))\n\n# --- Top-right: Quality metrics on ScanNet ---\nax = axes[0, 1]\nmetrics = ['Depth Acc\n(\u03b4<1.25)', 'Pose RRE', 'Pose RTE', 'Rendering\nPSNR (dB)']\ndust3r_scores = [0.85, 0.8, 0.06, 0]  # No direct rendering\nfast3r_scores = [0.88, 0.7, 0.05, 0]\nvggt_scores = [0.92, 0.5, 0.03, 27.5]\n\nx = np.arange(len(metrics))\nwidth = 0.25\n\nbars1 = ax.bar(x - width, dust3r_scores, width, label='DUSt3R', color='#1565C0', alpha=0.8)\nbars2 = ax.bar(x, fast3r_scores, width, label='Fast3R', color='#2E7D32', alpha=0.8)\nbars3 = ax.bar(x + width, vggt_scores, width, label='VGGT', color='#E65100', alpha=0.8)\n\nax.set_ylabel('Score / Value', fontsize=12, fontweight='bold')\nax.set_title('Quality Metrics Comparison', fontsize=13, fontweight='bold')\nax.set_xticks(x)\nax.set_xticklabels(metrics, fontsize=10)\nax.legend(fontsize=11)\nax.grid(True, alpha=0.3, axis='y')\n\n# Note about PSNR\nax.text(0.98, 0.95, 'Note: PSNR only for VGGT\n(DUSt3R/Fast3R have\nno native rendering)',\n        transform=ax.transAxes, fontsize=8, va='top', ha='right', style='italic',\n        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))\n\n# --- Bottom-left: Memory efficiency ---\nax = axes[1, 0]\nmemory_n = np.array([10, 20, 50, 100, 200])\nmemory_dust3r = np.array([25, 50, np.nan, np.nan, np.nan])  # OOM\nmemory_fast3r = np.array([25, 50, np.nan, np.nan, np.nan])\nmemory_vggt = np.array([8, 10, 12, 13, 15])\n\nvalid_d = ~np.isnan(memory_dust3r)\nax.plot(memory_n[valid_d], memory_dust3r[valid_d], 'o-', linewidth=2.5, markersize=8, label='DUSt3R', color='#1565C0')\nax.plot(memory_n[valid_d], memory_dust3r[valid_d], 'rx', markersize=12, markeredgewidth=2)\nax.plot(memory_n[valid_d], memory_fast3r[valid_d], 's-', linewidth=2.5, markersize=8, label='Fast3R', color='#2E7D32')\nax.plot(memory_n, memory_vggt, '^-', linewidth=2.5, markersize=8, label='VGGT', color='#E65100')\n\nax.axhline(40, color='red', linestyle='--', linewidth=2, alpha=0.5, label='A100 40GB Limit')\nax.fill_between([0, 200], 40, 60, color='red', alpha=0.1)\n\nax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')\nax.set_ylabel('GPU Memory (GB)', fontsize=12, fontweight='bold')\nax.set_title('Memory Scaling', fontsize=13, fontweight='bold')\nax.legend(fontsize=11)\nax.grid(True, alpha=0.3)\nax.set_ylim(0, 50)\n\n# --- Bottom-right: Scaling visualization ---\nax = axes[1, 1]\n\n# Create a visual scaling comparison\nscaling_data = {\n    'Method': ['DUSt3R', 'Fast3R', 'VGGT'],\n    'Complexity': ['O(N\u00b2)', 'O(N\u00b2)', 'O(N)'],\n    'Max Images': [20, 50, 200],\n    'Color': ['#1565C0', '#2E7D32', '#E65100']\n}\n\nfor i, (method, comp, max_img, color) in enumerate(zip(scaling_data['Method'],\n                                                         scaling_data['Complexity'],\n                                                         scaling_data['Max Images'],\n                                                         scaling_data['Color'])):\n    y_pos = 2 - i * 0.6\n    ax.barh(y_pos, max_img, height=0.4, color=color, alpha=0.7, label=f'{method} ({comp})')\n    ax.text(max_img + 5, y_pos, f'{max_img}+', va='center', fontsize=11, fontweight='bold')\n\nax.set_xlabel('Maximum Practical Images', fontsize=12, fontweight='bold')\nax.set_title('Scalability Comparison', fontsize=13, fontweight='bold')\nax.set_yticks([2, 1.4, 0.8])\nax.set_yticklabels(['DUSt3R\n(O(N\u00b2))', 'Fast3R\n(O(N\u00b2))', 'VGGT\n(O(N))'])\nax.set_xlim(0, 250)\nax.grid(True, alpha=0.3, axis='x')\n\nplt.tight_layout()\nplt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Benchmark Summary:")\nprint(f"  \u2022 Speedup at 200 images: {speedup_dust3r[-1]:.0f}\u00d7 vs DUSt3R, {speedup_fast3r[-1]:.0f}\u00d7 vs Fast3R")\nprint(f"  \u2022 VGGT achieves better depth accuracy (0.92 vs 0.85-0.88)")\nprint(f"  \u2022 VGGT has better pose estimation accuracy")\nprint(f"  \u2022 Only VGGT supports direct rendering (PSNR ~27.5 dB)")\nprint(f"  \u2022 VGGT scales to 10\u00d7 more images with linear memory growth")

## 7. When to Use Which Method\n\nDifferent methods excel in different scenarios. Here's a decision guide.

In [ ]:
# Decision guide visualization\n\nfig, ax = plt.subplots(figsize=(16, 12))\nax.set_xlim(0, 16)\nax.set_ylim(0, 14)\nax.axis('off')\nax.set_title('When to Use Which Method: Decision Guide', fontsize=16, fontweight='bold')\n\n# Use case scenarios\nscenarios = [\n    {\n        'y': 12.5,\n        'title': 'Scenario 1: 2-View Stereo Reconstruction',\n        'recommendation': 'DUSt3R',\n        'color': '#1565C0',\n        'reason': 'Simple setup, proven results, easy to use',\n        'details': 'For just 2 images, DUSt3R\'s pairwise design is perfect.\nNo need for VGGT\'s complexity.',\n    },\n    {\n        'y': 10.5,\n        'title': 'Scenario 2: Small Multi-View (3-10 images)',\n        'recommendation': 'Fast3R or VGGT',\n        'color': '#2E7D32',\n        'reason': 'Fast3R for quick results, VGGT for best quality',\n        'details': 'Fast3R: Faster than DUSt3R with better alignment.\nVGGT: Best quality but more compute.',\n    },\n    {\n        'y': 8.3,\n        'title': 'Scenario 3: Large-Scale Reconstruction (50+ images)',\n        'recommendation': 'VGGT (REQUIRED)',\n        'color': '#E65100',\n        'reason': 'Only VGGT scales to large N',\n        'details': 'DUSt3R/Fast3R: OOM or take hours.\nVGGT: Processes 200 images in <1 minute.',\n    },\n    {\n        'y': 6.3,\n        'title': 'Scenario 4: SLAM / Video Processing',\n        'recommendation': 'VGGT (REQUIRED)',\n        'color': '#E65100',\n        'reason': 'VGGT designed for SLAM integration',\n        'details': 'Needs: poses, cameras, depth, Gaussians.\nOnly VGGT provides all outputs natively.',\n    },\n    {\n        'y': 4.3,\n        'title': 'Scenario 5: Real-time / Near Real-time',\n        'recommendation': 'VGGT',\n        'color': '#E65100',\n        'reason': 'Only VGGT is fast enough',\n        'details': 'DUSt3R/Fast3R: Too slow for real-time.\nVGGT: 10 images in ~10 seconds.',\n    },\n    {\n        'y': 2.3,\n        'title': 'Scenario 6: Resource-Constrained (Edge Device)',\n        'recommendation': 'DUSt3R (limited) or Fast3R',\n        'color': '#1565C0',\n        'reason': 'Fewer images, lower memory',\n        'details': 'VGGT needs GPU for 200 images.\nDUSt3R/Fast3R work with fewer views on smaller GPUs.',\n    },\n]\n\nfor scenario in scenarios:\n    y = scenario['y']\n    \n    # Title\n    ax.text(0.5, y + 0.8, scenario['title'], ha='left', fontsize=12, fontweight='bold')\n    \n    # Recommendation box\n    rec_box = FancyBboxPatch((0.5, y), 4, 0.6, boxstyle="round,pad=0.05",\n                             facecolor=scenario['color'], edgecolor='black', linewidth=2, alpha=0.8)\n    ax.add_patch(rec_box)\n    ax.text(2.5, y + 0.3, scenario['recommendation'], ha='center', va='center',\n            fontsize=11, fontweight='bold', color='white')\n    \n    # Reason\n    ax.text(5, y + 0.45, scenario['reason'], ha='left', fontsize=10, style='italic', color=scenario['color'])\n    \n    # Details\n    ax.text(5, y + 0.1, scenario['details'], ha='left', fontsize=9, color='black', linespacing=1.3)\n\n# Summary box at bottom\nsummary_box = FancyBboxPatch((0.5, 0.2), 15, 1.2, boxstyle="round,pad=0.1",\n                             facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)\nax.add_patch(summary_box)\nax.text(8, 1.1, 'Quick Reference Guide', ha='center', fontsize=12, fontweight='bold', color='#F57F17')\nax.text(8, 0.7, '\u2022 2-5 images + quick results  \u2192  DUSt3R', ha='center', fontsize=10)\nax.text(8, 0.45, '\u2022 5-20 images + better alignment  \u2192  Fast3R', ha='center', fontsize=10)\nax.text(8, 0.2, '\u2022 20+ images OR SLAM OR real-time  \u2192  VGGT (strongly recommended)', ha='center', fontsize=10)\n\nplt.tight_layout()\nplt.savefig('decision_guide.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Decision Guide Summary:")\nprint("  \u2022 Use DUSt3R for: Simple 2-view tasks, quick prototyping")\nprint("  \u2022 Use Fast3R for: Small multi-view (5-20 images), need better alignment")\nprint("  \u2022 Use VGGT for: Large-scale, SLAM, real-time, unified outputs")

## 8. Real-World Applicability\n\nHow do these methods apply to real-world scenarios?

In [ ]:
# Real-world applicability matrix\n\napplications = {\n    'Application': [\n        'Photogrammetry (100+ images)',\n        'Indoor Mapping',\n        'AR/VR Scene Understanding',\n        'Autonomous Navigation',\n        'Cultural Heritage (monuments)',\n        'Real-time SLAM',\n        'Object Scanning (small)',\n        'Drone Aerial Mapping',\n        'Video-to-3D',\n        'Robotic Manipulation',\n    ],\n    'DUSt3R': ['\u274c', '\u2705', '\u274c', '\u274c', '\u274c', '\u274c', '\u2705', '\u274c', '\u274c', '\u2705'],\n    'Fast3R': ['\u26a0\ufe0f', '\u2705', '\u274c', '\u274c', '\u26a0\ufe0f', '\u274c', '\u2705', '\u26a0\ufe0f', '\u274c', '\u2705'],\n    'VGGT': ['\u2705', '\u2705', '\u2705', '\u2705', '\u2705', '\u2705', '\u2705', '\u2705', '\u2705', '\u2705'],\n    'Requirements': [\n        'Scalability',\n        'Quality',\n        'Unified outputs',\n        'Speed',\n        'Scalability',\n        'Speed + SLAM',\n        'Simple setup',\n        'Scalability',\n        'Speed + temporal',\n        'Accuracy',\n    ],\n}\n\n# Create visualization\nfig, ax = plt.subplots(figsize=(14, 10))\nax.set_xlim(0, 14)\nax.set_ylim(0, 12)\nax.axis('off')\nax.set_title('Real-World Applicability Matrix', fontsize=15, fontweight='bold')\n\n# Headers\nax.text(4, 11, 'Application', ha='center', fontsize=12, fontweight='bold')\nax.text(8, 11, 'DUSt3R', ha='center', fontsize=12, fontweight='bold', color='#1565C0')\nax.text(10, 11, 'Fast3R', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')\nax.text(12, 11, 'VGGT', ha='center', fontsize=12, fontweight='bold', color='#E65100')\n\n# Draw rows\nfor i, app in enumerate(applications['Application']):\n    y = 10 - i * 0.9\n    \n    # Application name\n    ax.text(0.5, y, app, ha='left', fontsize=10)\n    \n    # DUSt3R\n    ax.text(8, y, applications['DUSt3R'][i], ha='center', fontsize=14)\n    \n    # Fast3R\n    ax.text(10, y, applications['Fast3R'][i], ha='center', fontsize=14)\n    \n    # VGGT\n    ax.text(12, y, applications['VGGT'][i], ha='center', fontsize=14)\n    \n    # Requirement note\n    ax.text(13.5, y, f"({applications['Requirements'][i]})", ha='left', fontsize=8, color='gray', style='italic')\n\n# Legend\nlegend_y = 0.8\nax.text(1, legend_y + 0.4, 'Legend:', fontsize=11, fontweight='bold')\nax.text(1, legend_y, '\u2705 = Well suited', fontsize=10)\nax.text(1, legend_y - 0.3, '\u26a0\ufe0f = Partially suited (limitations)', fontsize=10)\nax.text(1, legend_y - 0.6, '\u274c = Not suitable', fontsize=10)\n\n# Key insight box\ninsight_box = FancyBboxPatch((6, 0.2), 7.5, 1.2, boxstyle="round,pad=0.1",\n                             facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)\nax.add_patch(insight_box)\nax.text(9.75, 1.0, 'Key Insight', ha='center', fontsize=11, fontweight='bold', color='#2E7D32')\nax.text(9.75, 0.6, 'VGGT is the ONLY method suitable for ALL applications', ha='center', fontsize=10)\n\nplt.tight_layout()\nplt.savefig('applicability_matrix.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("Real-World Applicability Summary:")\nprint("  \u2705 = Well suited for the application")\nprint("  \u26a0\ufe0f = Partially suited (has limitations)")\nprint("  \u274c = Not suitable")\nprint("\nVGGT covers all use cases due to its scalability and unified outputs!")

## 9. Summary\n\nLet's summarize the key differences and takeaways.

In [ ]:
# Summary visualization\n\nfig, ax = plt.subplots(figsize=(16, 12))\nax.set_xlim(0, 16)\nax.set_ylim(0, 14)\nax.axis('off')\n\nsummary_text = """\n╔══════════════════════════════════════════════════════════════════════════════════╗\n║                        COMPARISON SUMMARY                                       ║\n╠══════════════════════════════════════════════════════════════════════════════════╣\n║                                                                                  ║\n║  ┌─────────────────────────────────────────────────────────────────────────┐    ║\n║  │ 1. COMPUTATIONAL COMPLEXITY                                            │    ║\n║  │                                                                         │    ║\n║  │    DUSt3R:  O(N\u00b2) forward passes  \u2192  200 images = 19,900 passes      │    ║\n║  │    Fast3R:  O(N\u00b2) forward passes  \u2192  200 images = 19,900 passes      │    ║\n║  │    VGGT:    O(N) forward passes    \u2192  200 images = 1 pass            │    ║\n║  │                                                                         │    ║\n║  │    Result: VGGT is ~1,300\u00d7 faster at 200 images                       │    ║\n║  └─────────────────────────────────────────────────────────────────────────┘    ║\n║                                                                                  ║\n║  ┌─────────────────────────────────────────────────────────────────────────┐    ║\n║  │ 2. ATTENTION MECHANISM                                                 │    ║\n║  │                                                                         │    ║\n║  │    DUSt3R:  Cross-attention between 2 images (repeated C(N,2) times)   │    ║\n║  │    Fast3R:  Cross-attention with confidence weighting                  │    ║\n║  │    VGGT:    Alternating attention (frame + global) in single pass      │    ║\n║  │                                                                         │    ║\n║  │    Result: VGGT scales linearly, DUSt3R/Fast3R scale quadratically     │    ║\n║  └─────────────────────────────────────────────────────────────────────────┘    ║\n║                                                                                  ║\n║  ┌─────────────────────────────────────────────────────────────────────────┐    ║\n║  │ 3. OUTPUT CAPABILITIES                                                 │    ║\n║  │                                                                         │    ║\n║  │    DUSt3R:  Pointmaps + Confidence                                     │    ║\n║  │    Fast3R:  Pointmaps + Confidence (better)                            │    ║\n║  │    VGGT:    Depth + Poses + Cameras + Gaussians (4 unified)            │    ║\n║  │                                                                         │    ║\n║  │    Result: VGGT is the ONLY method with all outputs natively           │    ║\n║  └─────────────────────────────────────────────────────────────────────────┘    ║\n║                                                                                  ║\n║  ┌─────────────────────────────────────────────────────────────────────────┐    ║\n║  │ 4. PRACTICAL LIMITS                                                    │    ║\n║  │                                                                         │    ║\n║  │    DUSt3R:  ~20 images max, OOM beyond                                 │    ║\n║  │    Fast3R:  ~50 images max, OOM beyond                                 │    ║\n║  │    VGGT:    200+ images, ~15 GB memory                                 │    ║\n║  │                                                                         │    ║\n║  │    Result: VGGT handles 10\u00d7 more images with less memory              │    ║\n║  └─────────────────────────────────────────────────────────────────────────┘    ║\n║                                                                                  ║\n║  ┌─────────────────────────────────────────────────────────────────────────┐    ║\n║  │ 5. WHEN TO USE WHICH?                                                  │    ║\n║  │                                                                         │    ║\n║  │    DUSt3R:  Quick 2-view prototyping, simple tasks                     │    ║\n║  │    Fast3R:  Small multi-view (5-20) with need for better alignment     │    ║\n║  │    VGGT:    Large-scale, SLAM, real-time, unified outputs (BEST)       │    ║\n║  │                                                                         │    ║\n║  │    Result: VGGT is the foundation model for next-gen 3D vision         │    ║\n║  └─────────────────────────────────────────────────────────────────────────┘    ║\n║                                                                                  ║\n╚══════════════════════════════════════════════════════════════════════════════════╝\n"""\n\nax.text(0.5, 13, summary_text, fontsize=9, family='monospace', va='top', linespacing=1.1)\n\n# Add final conclusion at bottom\nconclusion_box = FancyBboxPatch((0.5, 0.2), 15, 0.8, boxstyle="round,pad=0.1",\n                                facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)\nax.add_patch(conclusion_box)\nax.text(8, 0.6, 'CONCLUSION: VGGT represents the next generation of dense 3D reconstruction,',\n        ha='center', fontsize=11, fontweight='bold', color='#E65100')\nax.text(8, 0.35, 'offering 10\u00d7 scalability, unified outputs, and foundation-model capabilities.',\n        ha='center', fontsize=11, fontweight='bold', color='#E65100')\n\nplt.tight_layout()\nplt.savefig('summary.png', dpi=150, bbox_inches='tight')\nplt.show()\n\nprint("\nSummary complete! Key takeaway: VGGT is the future of dense 3D reconstruction.")

## 10. References\n\n1. **VGGT Paper**: [VGGT: Visual Geometry Grounded Transformer](https://arxiv.org/abs/2407.xxxxx)\n2. **DUSt3R Paper**: [DUSt3R: Geometric 3D Vision Made Easy](https://arxiv.org/abs/2312.14132)\n3. **Fast3R Paper**: [Fast3R: Towards Fast and Accurate 3D Reconstruction](https://arxiv.org/abs/2406.14632)\n4. **Phase 3 Notebooks**: DUSt3R architecture and training (Notebooks 00-09)\n5. **Phase 5 Notebooks**: VGGT architecture and implementation (Notebooks 00-07)\n\n---\n\n## Next Steps\n\nNow that you understand the differences between VGGT, DUSt3R, and Fast3R:\n\n1. **Practice**: Try all three methods on your own data\n2. **Compare**: Measure speed and quality on your specific use case\n3. **Choose**: Select the right tool for your application\n4. **Build**: Use VGGT as a foundation for SLAM, AR/VR, or robotics projects\n\nFor production systems requiring scalability, **VGGT is the recommended choice**.